# Watt The Hack - Local Playtester Sandbox (v0.1.3)

Welcome to the playtester sandbox! This notebook allows you to test controllers against hidden scenarios *entirely on your local machine*.
You don't need to ZIP files or wait for cloud evaluation. It runs instantly and prints the exact same metrics and cost breakdown the cloud judge uses.

## 1. Install the Engine & Upload Scenarios

Because scenarios contain hidden judging data, they are not public. 
**You must upload the `scenarios.zip` file provided by the organizers to your Colab session now.**

Run this cell to install the public engine and extract your uploaded scenarios.

In [20]:
import os
import zipfile
from pathlib import Path

# 1. Install engine — repo devs get local code; playtesters get public package
_repo_root = Path("..").resolve() if (Path("..") / "pyproject.toml").is_file() else Path(".").resolve()
if (_repo_root / "pyproject.toml").is_file():
    %pip install -e {_repo_root}
else:
    %pip install git+https://github.com/AaronEliasZachariah/Watt-The-Hack-Engine-Public.git

# 2. Scenarios: organizers' zip OR this repo's scenarios/ folder
if Path("scenarios.zip").exists():
    with zipfile.ZipFile("scenarios.zip", "r") as zip_ref:
        zip_ref.extractall("scenarios_dir")
    os.environ["SCENARIOS_DATA_DIR"] = str(Path("scenarios_dir/scenarios").resolve())
    print(f"\n✅ Scenarios from zip → {os.environ['SCENARIOS_DATA_DIR']}")
else:
    for candidate in (Path("scenarios"), Path("../scenarios")):
        if candidate.is_dir():
            os.environ["SCENARIOS_DATA_DIR"] = str(candidate.resolve())
            print(f"\n✅ Using repo scenarios → {os.environ['SCENARIOS_DATA_DIR']}")
            break
    else:
        os.environ.pop("SCENARIOS_DATA_DIR", None)
        print("\n⚠️ No scenarios.zip and no scenarios/ folder found. Upload scenarios.zip (Colab) or run from the Watt-The-Hack repo.")

# Set SCENARIOS_DATA_DIR *before* importing watt_the_hack.data_loaders.scenarios in later cells.

Defaulting to user installation because normal site-packages is not writeable
Obtaining file:///C:/Users/aaron/GitHub/Uni/Watt-The-Hack
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for watt-the-hack (pyproject.toml): started
  Building editable for watt-the-hack (pyproject.toml): finished with status 'done'
  Created wheel for watt-the-hack: filename=watt_the_hack-0.1.0-0.editable-py3-none-any.whl size=4097 sha256=ecdd2360ea20c3804827e385882e0845ace10a22d10abbab8878c55855f5d66b
  Stored in directory: C:\Users\aaron\AppData\

## 2. Write your Agent

Put your logic in `strategy.py` for cloud submit. Two shapes are supported (pick one in `metadata.json`):

- **Function:** `def controller(state): return {...}` → `"function_name": "controller"`
- **Class:** `class Strategy:` with `step(state)` → `"class_name": "Strategy"` (optional `plan` / `replan`)

The class is not passed to the engine directly — the evaluator builds a callable that calls your methods. Below is the class form for agentic controllers (LLM in `plan()`, fast logic in `step()`).

In [21]:
import os
#import openai

class Strategy:
    def __init__(self):
        # Local evaluation allows you to use your real API keys!
        self.api_key = os.environ.get("OPENAI_API_KEY", "dummy-key")
        self.target_soc = 0.5
        
    def plan(self, state: dict) -> dict:
        """
        Called ONCE at the start. 
        Read the briefing, make an LLM call, and decide your strategy!
        """
        briefing = state.get("qualitative_briefing", "")
        print(f"\n[Agent] Reading briefing: {briefing}")
        
        # Example strategy selection:
        self.target_soc = 0.8
        print(f"[Agent] Decided on target SOC: {self.target_soc}\n")
        
        return {"agent_plan": {"target_soc": self.target_soc}}

    def replan(self, state: dict, alerts: list) -> dict:
        """
        Called mid-run if a 'qualitative_alert' event fires (e.g. weather changing).
        """
        print(f"\n[Agent] ALERT! {alerts}")
        self.target_soc = 0.2
        return {"agent_plan": {"target_soc": self.target_soc}}

    def step(self, state: dict) -> dict:
        """
        Called every 15-minute timestep. 
        NO LLM calls here, just execute the plan quickly!
        """
        plan = state.get("agent_plan", {})
        target = plan.get("target_soc", self.target_soc)
        current_soc = state.get("soc", 0.0)
        
        flow = 0.0
        if current_soc < target - 0.05:
            flow = -50.0  # charge battery
        elif current_soc > target + 0.05:
            flow = 50.0   # discharge battery
            
        return {
            "battery_flow_kw": flow,
            "emergency_generator": 0.0,
            "curtail_solar": 0.0,
            "fcas_reserve_kw": 0.0
        }

## 3. Run the Backtest Locally

Pick a `scenario_id` to evaluate against. This code mimics exactly what the cloud evaluator does behind the scenes, including hooking up `.plan()` and tracking your detailed cost breakdown.

In [22]:
import json
from watt_the_hack.engine.engine import Engine
from watt_the_hack.simulation.runner import run_simulation
from watt_the_hack.data_loaders.scenarios import load_scenario

# Use the "id" inside the JSON file — for duck_curve.json that is just "duck_curve"
SCENARIO_ID = "duck_curve"

from pathlib import Path
from watt_the_hack.data_loaders.scenarios import find_scenario_by_id, list_scenarios, scenarios_root

_root = scenarios_root()
print(f"Scenarios root: {_root}")
scenario_path = find_scenario_by_id(SCENARIO_ID)
if scenario_path is None:
    # Optional: load by JSON file path instead of id
    for candidate in (
        Path(SCENARIO_ID),
        _root / SCENARIO_ID,
        Path("..") / SCENARIO_ID,
    ):
        if candidate.is_file():
            scenario_path = candidate.resolve()
            break
if scenario_path is None:
    available = [s["id"] for s in list_scenarios(include_judging=True)]
    raise FileNotFoundError(
        f"No scenario {SCENARIO_ID!r}. Available ids: {available}\n"
        "(Use the JSON \"id\" field, e.g. duck_curve — not synthetic/duck_curve.json.)"
    )
print(f"Loading: {scenario_path}")

spec, initial_state = load_scenario(scenario_path)
strategy = Strategy()

# Cloud runtime: Strategy() once, optional plan(), then step() every timestep.
# run_simulation expects a single callable — we build that wrapper locally.
if hasattr(strategy, "plan"):
    plan_result = strategy.plan(Engine.controller_view(initial_state))
    if isinstance(plan_result, dict):
        initial_state.update(plan_result)

def controller_wrapper(view):
    alerts = view.get("alerts", [])
    if alerts and hasattr(strategy, "replan"):
        update = strategy.replan(view, alerts)
        if isinstance(update, dict):
            initial_state["agent_plan"] = {
                **initial_state.get("agent_plan", {}),
                **update.get("agent_plan", {}),
            }
    return strategy.step(view)

n_steps = len(initial_state["_profiles_full"]["demand"])
print(f"Evaluating {n_steps} steps...")
result = run_simulation(
    controller=controller_wrapper,
    initial_state=initial_state,
    steps=n_steps,
)

# 4. Aggregate and display exactly like the cloud
all_breakdowns = [out.get("cost_breakdown", {}) for out in result["outputs"]]
agg_breakdown = {}
for bd in all_breakdowns:
    for k, v in bd.items():
        agg_breakdown[k] = agg_breakdown.get(k, 0.0) + float(v)

print("\n✅ Evaluation Complete!")
print(f"\nFinal Score (Cost): ${result['metrics']['final_score']:.2f}")
print("\n--- Cost Breakdown ---")
print(json.dumps(agg_breakdown, indent=2))


Scenarios root: c:\Users\aaron\GitHub\Uni\Watt-The-Hack\notebooks\scenarios_dir\scenarios


FileNotFoundError: No scenario 'duck_curve'. Available ids: []
(Use the JSON "id" field, e.g. duck_curve — not synthetic/duck_curve.json.)